# Two-Phase Training — LoRA SFT → DPO

**Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Device:** Apple MPS (Mac M-series) / CPU fallback  
**Libraries:** HuggingFace `transformers`, `peft`, `trl`

## Pipeline Overview
```
Phase 1 — SFT with LoRA
  └── Load pre-trained Qwen2.5-0.5B-Instruct
  └── Attach LoRA adapters (rank=8)
  └── Train on SFT dataset with SFTTrainer
  └── Save SFT model → models/sft_model/

Phase 2 — DPO
  └── Load SFT model (now the reference + policy model)
  └── Attach fresh LoRA adapters for DPO
  └── Train on DPO dataset with DPOTrainer
  └── Save final model → models/dpo_model/
  └── Push to HuggingFace Hub
```

> **Prerequisites:** Run `data_preparation.ipynb` first.

# PHASE 2 Training - DPO

## Istallations and Device Setup

In [1]:
import torch
import gc
from pathlib import Path
from datasets import load_from_disk, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trl import DPOTrainer, DPOConfig

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Device    : {DEVICE}")
print(f"PyTorch  : {torch.__version__}")

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_DIR = Path("../data")
MODELS_DIR = Path("../models")
SFT_MODEL_PATH = MODELS_DIR / "sft_model/sft_final"
DPO_MODEL_PATH = MODELS_DIR / "dpo_model"


Device    : mps
PyTorch  : 2.11.0


## Loading the DPO Dataset 

In [2]:
dpo_data  = load_from_disk(str(DATA_DIR / "dpo_dataset"))
test_data = load_from_disk(str(DATA_DIR / "test_dataset"))

print(f"DPO  train : {len(dpo_data['train']):,}")
print(f"DPO  val   : {len(dpo_data['validation']):,}")
print(f"Test       : {len(test_data):,}")
print(f"\nDPO columns  : {dpo_data['train'].column_names}")

DPO  train : 1,687
DPO  val   : 188
Test       : 810

DPO columns  : ['prompt', 'chosen', 'rejected']


In [3]:
# Loading Tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    SFT_MODEL_PATH, trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded from : {SFT_MODEL_PATH}")

Tokenizer loaded from : ../models/sft_model/sft_final


In [4]:
SYSTEM_PROMPT=""" You are a knowledgeable and empathetic customer support assistant. 
Your role is to help customers resolve their issues quickly and accurately. 
Always be polite, clear, and solution-focused."""

## Loading the SFT Model as DPO Starting Policy

In [ ]:
if DEVICE in ["cuda", "mps"]:
    dtype = torch.float16
else:
    dtype = torch.float32

print(f"Loading SFT model from : {SFT_MODEL_PATH}")
base_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=dtype
)
base_model.config.use_cache = False

Loading SFT model from : ../models/sft_model/sft_final


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

In [6]:
print("Mounting SFT Adapter...")
policy_model = PeftModel.from_pretrained(
    base_model,
    str(SFT_MODEL_PATH),
    is_trainable=True # We want to train the adapter during DPO fine-tuning
)
policy_model.to(DEVICE)

policy_model.print_trainable_parameters()
print("\nPolicy Model Ready for DPO fine-tuning!\nSFT Weights are the Starting Point for DPO Training.")

Mounting SFT Adapter...


/Users/anjishnu_mukherjee/Documents/CODING/TMLC_Projects/Cust-Supp-faq-HFModel/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

Policy Model Ready for DPO fine-tuning!
SFT Weights are the Starting Point for DPO Training.


## DPO Training 

In [7]:
dpo_args = DPOConfig(
    output_dir=str(DPO_MODEL_PATH),

    # DPO specific
    beta=0.1,                  # KL penalty — 0.1 keeps policy close to SFT ref
    loss_type="sigmoid",       # standard DPO loss
    max_length=256,

    # Schedule
    num_train_epochs=1,
    per_device_train_batch_size=1,     # DPO holds policy + ref simultaneously
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,     # effective batch = 8

    # Optimizer
    learning_rate=5e-5,                # lower than SFT — fine-grained adjustment
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="adamw_torch",
    weight_decay=0.01,

    # Evaluation
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=25,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # MPS safe settings
    gradient_checkpointing=False,
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False,

    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=None,            # None → DPOTrainer auto-creates frozen ref copy
    args=dpo_args,
    train_dataset=dpo_data["train"],
    eval_dataset=dpo_data["validation"], 
    processing_class=tokenizer,
)

print("DPOTrainer ready")
print(f"Train samples : {len(dpo_data['train']):,}")
print(f"Val samples   : {len(dpo_data['validation']):,}")
print(f"Beta          : 0.1")
print(f"Eff. batch    : {1 * 8}")
print(f"Steps/epoch   : ~{len(dpo_data['train']) // (1 * 8)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


DPOTrainer ready
Train samples : 1,687
Val samples   : 188
Beta          : 0.1
Eff. batch    : 8
Steps/epoch   : ~210


In [8]:
print("Starting PHASE 2 DPO Training")
print("═" * 55)

dpo_result = dpo_trainer.train()

print("═" * 55)
print(f"DPO complete")
print(f"Train loss : {dpo_result.training_loss:.4f}")
print(f"Runtime    : {dpo_result.metrics.get('train_runtime', 0) / 60:.1f} min")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting PHASE 2 DPO Training
═══════════════════════════════════════════════════════


Step,Training Loss,Validation Loss
50,0.047895,0.017833
100,0.010269,0.010636
150,0.007837,0.006574
200,0.009198,0.006326
211,0.009198,0.006343


═══════════════════════════════════════════════════════
DPO complete
Train loss : 0.0700
Runtime    : 31.6 min


## Evaluation on Test Cases

In [9]:
# Format test set for evaluation (prompt + reference answer)
def format_test_sample(example):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
    ]
    
    # Prompt used as model context during evaluation
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    # Reference answer to score under teacher forcing
    response_text = example["response"].strip()
    full_text = prompt_text + response_text + tokenizer.eos_token
    
    example["prompt_text"] = prompt_text
    example["response_text"] = response_text
    example["full_text"] = full_text
    return example

test_formatted = test_data.map(format_test_sample)
test_for_eval = test_formatted.select_columns(["prompt_text", "full_text"])

print(f"Test set formatted : {len(test_formatted):,} samples")
print(
    f"Eval-ready subset  : {len(test_for_eval):,} samples, "
    f"columns: {test_for_eval.column_names}"
)

Map:   0%|          | 0/810 [00:00<?, ? examples/s]

Test set formatted : 810 samples
Eval-ready subset  : 810 samples, columns: ['prompt_text', 'full_text']


In [10]:
from torch.utils.data import DataLoader

print("Tokenizing test set for evaluation...")

def tokenize_for_eval(example):
    full_enc = tokenizer(
        example["full_text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    prompt_enc = tokenizer(
        example["prompt_text"],
        truncation=True,
        max_length=512,
        padding=False,
    )

    labels = full_enc["input_ids"].copy()
    prompt_len = min(len(prompt_enc["input_ids"]), len(labels))

    # Ignore prompt tokens: evaluate loss only on assistant response tokens
    for i in range(prompt_len):
        labels[i] = -100

    # Ignore padded positions in loss
    for i, mask_val in enumerate(full_enc["attention_mask"]):
        if mask_val == 0:
            labels[i] = -100

    full_enc["labels"] = labels
    return full_enc

test_tokenized = test_for_eval.map(
    tokenize_for_eval,
    remove_columns=["prompt_text", "full_text"],
    desc="Tokenizing test set"
)
test_tokenized.set_format("torch")

policy_model.eval()
dataloader = DataLoader(test_tokenized, batch_size=2)

total_loss = 0
total_steps = 0

print("Evaluating on test set...")
with torch.no_grad():
    for batch in dataloader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = policy_model(**batch)
        total_loss += outputs.loss.item()
        total_steps += 1

avg_loss = total_loss / max(total_steps, 1)
perplexity = torch.exp(torch.tensor(avg_loss)).item()

print("\n── Test Evaluation Results ──────────────────────────")
print(f"   test_loss       : {avg_loss:.4f}")
print(f"   test_perplexity : {perplexity:.4f}")

Tokenizing test set for evaluation...


Tokenizing test set:   0%|          | 0/810 [00:00<?, ? examples/s]

Evaluating on test set...

── Test Evaluation Results ──────────────────────────
   test_loss       : 1.6080
   test_perplexity : 4.9929


## Inference Check

In [11]:
def generate_response(model, tokenizer, user_query, max_new_tokens=256, temperature=0.0):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Deterministic decode by default for repeatable notebook checks.
    do_sample = temperature > 0

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=0.9 if do_sample else None,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated = outputs[0][prompt_len:]  # Skip prompt tokens

    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [12]:
test_queries = [
    "I want to cancel my subscription. How do I do that?",
    "I haven't received my order and it's been 10 days.",
    "How do I reset my account password?",
]

print("-- DPO Policy Model Outputs --")
for q in test_queries:
    print(f"\nUser  : {q}")
    print(
        "Model : "
        f"{generate_response(policy_model, tokenizer, q, max_new_tokens=192, temperature=0.0)}"
    )
    print("-" * 55)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


-- DPO Policy Model Outputs --

User  : I want to cancel my subscription. How do I do that?
Model : Thank you for wanting to cancel your subscription and wishing to proceed with cancelling it. To cancel your subscription, follow these steps:

1. Log in to your account on our platform.
2. Navigate to the subscription management section or dashboard where you can view your current subscriptions.
3. Locate the specific subscription you wish to cancel and click on it.
4. Look for an option labeled "Cancel Subscription" or similar and select it.
5. If prompted, confirm the cancellation by clicking on the confirmation button.

If you encounter any difficulties or have further questions during this process, please don't hesitate to let me know. I'm here to assist you every step of the way in canceling your subscription and ensuring a smooth experience.
-------------------------------------------------------

User  : I haven't received my order and it's been 10 days.
Model : I understand that 

## Saving the Model

In [13]:
DPO_FINAL_PATH = DPO_MODEL_PATH / "dpo_final"
DPO_FINAL_PATH.mkdir(parents=True, exist_ok=True)

dpo_trainer.save_model(str(DPO_FINAL_PATH))
tokenizer.save_pretrained(str(DPO_FINAL_PATH))

print(f"DPO model and tokenizer saved to {DPO_FINAL_PATH}")

DPO model and tokenizer saved to ../models/dpo_model/dpo_final
